In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# create our numpy random number generator
rng = np.random.default_rng(42)

# Classical Statistical Inference

We now have enough machinery to remove some of the mystery surrounding classical statistical inference. If you're like most people, p-values and null hypotheses were a bit mysterious when you first learned about them. If you think about them in terms of sampling distributions, which we can simulate, hopefully they will become clearer.

The key steps are:
1. Make some assumption about the process by which your sample of data was generated (the null hypothesis)
2. Simulate the sampling distribution for that data generation process
    - Simulate a bunch of alternative data samples based on that sample
    - Compute the sample statistic for each
    - Plot the density (frequency) of different outcomes
3. Assess how surprising your actual sample statistic is, with reference to that sampling distribution
    - Quantify that as the p-value
    - If the p-value is below some threshold (the significance threshold), **reject the original null hypothesis**

### Scenario: A Reaction Time Experiment

Let's consider a pretty common scenario for statistical inference. You've conducted a psychology experiment, with 40 subjects, to test if drinking one beer is enough to noticeably degrade their reaction times.

We test reaction times using a computer apparatus. Images and numbers appear on the screen; When a red circle shows up, the person hits a button. The amount of time elapsed from the red circle appearing until they hit the button is their reaction time. 

For each subject, we measure their reaction time (the Before measure), then have them drink one beer, then test their reaction time again (the After measure). 

We want to know whether the After measure is generally bigger than the Before measure. But there could be some random noise. For some people, they might be a little distracted during the Before test, and for others during the After test.

Let's look at an imaginary dataframe showing the results of this hypothetical experiment.



In [ ]:
exp_results = pd.read_csv('data/reaction_times.csv')

In [ ]:
exp_results.head()

In [ ]:
exp_results['increase'] = exp_results['after'] - exp_results['before']

In [ ]:
exp_results['increase'].hist()

In [ ]:
exp_results['increase'].describe()

### Is the Increased Reaction Time Statistically Significant?

OK. There's an increased reaction time, on average, of about 0.06 seconds.

How confident are we, however, that this difference is "real"? Might it be possible that on average, in the whole population, people's reactions don't increase after one beer, but we happened to get a sample of 40 people that doesn't reflect the overall population or unusual reaction times for them (maybe their cell phone started ringing during the test)? 

Of course, that is possible. But how plausible it?

With Null Hypothesis Statistical Testing, we actually ask a slightly different question. *Suppose* that one beer has absolutely no effect on average across the population. How surprised would be to see a result as extreme as we saw, 0.06 seconds?

# Articulating the Null Hypothesis as a Distribution (Data Generation Process)

Given our dataframe `exp_results`, I'm going to show you two different ways that we could formulate the null hypothesis in a way that allows us to generate lots of simulated experiments.

#### 1a. Parametric Assumption: Change in Reaction Time Follows a Normal Distribution

Our first approach will be to assume that each person's change in reaction times from Before to After is an independent draw from a Normal Distribution.

We will assume it has mean 0 (the null hypothesis is that the mean difference is 0). 
For the variance of that assumed null distribution, we will make our best guess, which is the empirical variance we saw in our `exp_results` dataset.



In [ ]:
actual_mean = exp_results['increase'].mean()
actual_std = exp_results['increase'].std()
actual_mean, actual_std

One simulated sample of 40

In [ ]:
rng.normal(0, actual_std, 40)

#### 2a. Simulate Sampling Distribution

Now we will simulate lots of samples of size 40, and compute the mean of each sample.

In [ ]:
n_simulations=50000

Remember these two useful functions from previous notebooks

In [ ]:
# simulate the distribution of some sample statistic, given a simulator for the underlying distribution
def simulate_sampling_distribution(
    underlying_distribution_simulator, sample_statistic, n_simulations=1000, sample_size=100
):
    samples = [underlying_distribution_simulator(sample_size) for _ in range(n_simulations)]
    return pd.DataFrame(
        {
            "statistic": [sample_statistic(sample) for sample in samples],
        }
    )

# plot the histogram of a dataset generated from a continuous distribution, by binning the data
def plot_continuous(data, ax, colname="statistic", num_bins=100, min_x=None, max_x=None):
    sns.histplot(data[colname], ax=ax, stat="probability")

    # Set the range of the x-axis if min_x and max_x are provided
    if min_x is not None and max_x is not None:
        ax.set_xlim(min_x, max_x)

    ax.set_ylabel("Probability")

In [ ]:
sample_means_a = simulate_sampling_distribution(
    lambda n: rng.normal(0, actual_std, n),
    lambda x: x.mean(),
    n_simulations=n_simulations,
    sample_size=40,
)

In [ ]:
sample_means_a.describe()

#### 3a. Assess Surprise (Significance)

Compute the p-value. What fraction of the sample means are farther from the true mean than the observed mean's distance?

In [ ]:
# two-sided p-value, so check if absolute value of the difference
p_value_a = (np.abs(sample_means_a["statistic"]) >= abs(actual_mean)).mean()
p_value_a

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
plt.subplots_adjust(wspace=0.5)
plot_continuous(sample_means_a, axes[0], num_bins=100)

# add a vertical line at the actual mean
axes[0].axvline(actual_mean, color="red", linestyle="--")
axes[0].set_title('Parameteric Null Hypothesis')

# A Different Simulation of the Null Hypothesis
#### 1b. Nonparametric Null Hypothesis Using Permutation Sampling

Again imagine that there is actually no difference in the before and after reaction times, on average. That is, different people may have different reaction times, but for each person, our null hypothesis is that we are getting two independent draws from whatever that person's reaction time is.

What we actually saw in our dataset is the two draws. But if those were independent draws from the same distribution, we could imagine simulating a different outcome by flipping a coin for whether those two draws are treated as the before and after, or the reverse.

This is a version of what is called permutation sampling, or shuffling.

In [ ]:
def shuffle(df):
    # for each row, get a coin flip for whether to swap Before and After
    flips = rng.integers(0, 2, len(df))
    # if flip is 1, swap the values
    shuffled_df = pd.DataFrame({
        'before': np.where(flips, df['after'], df['before']),
        'after': np.where(flips, df['before'], df['after']),
    })
    return shuffled_df['after'] - shuffled_df['before']

#### 2b. Simulate Sampling Distribution

In [ ]:
sample_means_b = simulate_sampling_distribution(
    lambda n: shuffle(exp_results),
    lambda x: x.mean(),
    n_simulations=n_simulations,
    sample_size=40,
)

In [ ]:
sample_means_b.describe()

#### 3b. Assess Surprise (Significance)

Compute the p-value. What fraction of the sample means are farther from the true mean than the observed mean's distance?

In [ ]:
# two-sided p-value, so check if absolute value of the difference
p_value_b = (np.abs(sample_means_b["statistic"]) >= abs(actual_mean)).mean()
p_value_b

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
plt.subplots_adjust(wspace=0.5)
plot_continuous(sample_means_a, axes[0], num_bins=100)
plot_continuous(sample_means_b, axes[1], num_bins=100)

# add a vertical line at the actual mean
axes[0].axvline(actual_mean, color="red", linestyle="--")
axes[1].axvline(actual_mean, color="red", linestyle="--")
axes[0].set_title('Parameteric Null Hypothesis')
axes[1].set_title('Non-Parametric: Permutation')

# Nonparametric Estimation of Confidence Intervals using Bootstrap Sampling

We can do something analogous to construct a "confidence interval" around the observed mean.

Here, instead of simulating other samples of size 40 from a distribution based on the null hypothesis, we simulate samples of size 40 by bootstrap resampling of the observed data.

Again, for each sample we compute the mean. The interval covering 95% of the sample means is a 95% confidence interval. 

While it won't come out exactly the same, since we are using a slightly different procedure for estimating the sampling distribution, we the proportion of values in this sampling distribution that are farther away than 0 from the observed mean should be about the same as the p-values calculated based on a null hypothesis.


In [ ]:
def bootstrap_resample(df, n_people):
    return rng.choice(df['increase'], n_people, replace=True)

#### 2b. Simulate Sampling Distribution

In [ ]:
sample_means_c = simulate_sampling_distribution(
    lambda n: bootstrap_resample(exp_results, n),
    lambda x: x.mean(),
    n_simulations=n_simulations,
    sample_size=40,
)

In [ ]:
sample_means_c.describe()

#### 3b. Assess Surprise (Significance)

Compute the p-value. What fraction of the sample means are farther from the true mean than 0 is?

In [ ]:
p_value_c = (np.abs(sample_means_c["statistic"] - actual_mean) > actual_mean).mean()
p_value_c

Compute the 95% confidence interval

In [ ]:
# what range covers 95% of the bootstrap samples?
lower, upper = np.percentile(sample_means_c['statistic'], [2.5, 97.5])
lower, upper

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
plt.subplots_adjust(wspace=0.5)
plot_continuous(sample_means_a, axes[0], num_bins=100)
plot_continuous(sample_means_b, axes[1], num_bins=100)
plot_continuous(sample_means_c, axes[2], num_bins=100)

# add a vertical line at the actual mean
axes[0].axvline(actual_mean, color="red", linestyle="--")
axes[1].axvline(actual_mean, color="red", linestyle="--")
axes[2].axvline(lower, color="red", linestyle="--")
axes[2].axvline(upper, color="red", linestyle="--")
axes[0].set_title('Parameteric Null Hypothesis')
axes[1].set_title('Non-Parametric: Permutation')
axes[2].set_title('Bootstrap CI Estimation')

There's a connection between the 95% confidence intervals and hypothesis tests:
- If the 95% confidence interval includes the value that the null hypothesis assumes, then the p-value with a hypothesis test will be above 5%
    - Here we see that the bootstrap confidence interval includes 0
    - Thus, you should not reject the null hypothesis
- Conversely, if the 95% confidence interval doesn't include that, then the p-value with a hypothesis test will be below 5%
    - Then, you would reject the null hypothesis

# So, Does One Beer Make You Slower?
(Remember, this is fictitious data, so don't make any real-life decisions based on it!)

It *looked* like one beer made people slower, by .06 seconds on average. 

But we failed to reject the null hypothesis. 
- Even if it there were no effect, and the before and after were two indendepent draws from each person's personal reaction time, it wouldn't be so surprising to see an average difference of .06 seconds.
    - It would happen in about 13% of all samples of 40 people
- Relatedly, the bootstrap process includes 0 within the 95% confidence interval

But beware, failure to reject the null hypothesis doesn't mean you should "accept" it! 
- My starting intuition is that one beer does have some effect on most people's reaction times. 
- This dataset, if it were real, would increase that belief! It certainly wouldn't make me believe that the null hypothesis is true.
